# Scoring Engine — Weight Evaluation

PDF §6.2 acceptance criterion: "scoring weights are documented, externalized to YAML, and adjustable without changing any code". This notebook quantifies the impact of each weight on a small set of manually-labeled queries.

Approach:
1. Build a synthetic dataset of `(BusinessListing, expected_rank)` tuples for a representative category.
2. Score under several weight configurations.
3. Report Spearman rank correlation between the engine's ordering and the expected ordering.

Run with: `jupyter notebook backend/notebooks/scoring_eval.ipynb`.

In [ ]:
import sys, os
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
os.environ.setdefault('CACHE_DIR', str(Path.cwd() / '.eval_cache'))

from app.models.business import BusinessListing, ReviewData
from app.models.intent import ParsedIntent
from app.modules.scoring_engine import ScoringEngine, _compute_composite, _DEFAULT_WEIGHTS

engine = ScoringEngine()
print('Default weights:', _DEFAULT_WEIGHTS)

In [ ]:
def mk(name, rating, count, pos_pct, recency):
    return BusinessListing(
        id=name, name=name, category='restaurant', source='overpass',
        review_data=ReviewData(
            average_rating=rating,
            total_reviews=count,
            positive_percentage=pos_pct,
            recency_score=recency,
        ),
    )

# Curated set with a clearly-correct ordering: A > B > C > D > E
candidates = [
    mk('A — best',     rating=4.8, count=520, pos_pct=92.0, recency=0.95),
    mk('B — very good',rating=4.5, count=210, pos_pct=85.0, recency=0.80),
    mk('C — solid',    rating=4.2, count=110, pos_pct=78.0, recency=0.60),
    mk('D — middling', rating=3.8, count=45,  pos_pct=60.0, recency=0.40),
    mk('E — weak',     rating=3.0, count=8,   pos_pct=45.0, recency=0.20),
]
expected_order = ['A — best','B — very good','C — solid','D — middling','E — weak']

In [ ]:
intent = ParsedIntent(category='restaurant', location='near_me', raw_query='best restaurants')

def evaluate(weights):
    scored = sorted(candidates, key=lambda b: _compute_composite(b, weights), reverse=True)
    actual = [b.name for b in scored]
    # Simple agreement metric: count of positions matching expected order
    matches = sum(1 for i, n in enumerate(actual) if n == expected_order[i])
    return actual, matches

configs = {
    'default (40/30/20/10)': _DEFAULT_WEIGHTS,
    'rating-heavy (70/15/10/5)': {'star_rating': 0.70, 'review_count': 0.15, 'sentiment_score': 0.10, 'recency_signal': 0.05},
    'volume-heavy (15/55/20/10)': {'star_rating': 0.15, 'review_count': 0.55, 'sentiment_score': 0.20, 'recency_signal': 0.10},
    'recency-heavy (20/20/20/40)': {'star_rating': 0.20, 'review_count': 0.20, 'sentiment_score': 0.20, 'recency_signal': 0.40},
}
for name, w in configs.items():
    order, score = evaluate(w)
    print(f'{name:35s}  matches={score}/5  order={order}')

## Interpretation

Each weight config should produce the same ordering on this dataset because the signals are co-monotonic. If a config diverges from `A → E`, that's the weight value being miscalibrated relative to the saturation point of the underlying signal (e.g. `review_count` saturates at 100 reviews — increasing weight past a point yields no additional signal).